In [ ]:
#!/usr/bin/env python3
"""
two_for_one_mallows_hpc.py
==========================

Parallel evaluation of two_for_one() over all (accomplice, manipulator) pairs
for each row in 'mallows_gain.csv'. The result structure matches your
sequential script:

For each CSV row:
    instance_results = [
        [   # manipulator = 1
            [benefit, new_matching, accomplice_pref_after, beneficiary_pref_after],  # accomplice = 1
            ...
          ],
        [   # manipulator = 2
            ...
        ],
        ...
    ]

The script stores a list of these instance_results (one per row) in the
'two_for_one_results' column and writes back to 'mallows_gain_two_for_one.csv'.

Submit as:
    python3 -u two_for_one_mallows_hpc.py
"""

import os, sys, time, threading, itertools, copy, ast
import multiprocessing as mp
from pathlib import Path
import pandas as pd

# ----------------------------------------------------------------------
# 0.  Config + friendly BLAS threading caps
# ----------------------------------------------------------------------
CSV_NAME = "mallows_gain.csv"                       # input file
OUT_CSV_NAME = "mallows_gain_two_for_one.csv"       # output file
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

# ----------------------------------------------------------------------
# 1.  Stable matching + manipulation generators
# ----------------------------------------------------------------------
def gale_shapley(mpref, wpref, strategic_woman=0):
    n = len(mpref)
    curr_match_count = 0
    last_rejected_by_rank_what = [0 for _ in range(n)]
    men_matchings = [0 for _ in range(n)]
    women_matchings = [0 for _ in range(n)]
    proposals_to_strategic_woman = set()  # only used if strategic_woman != 0

    while curr_match_count < n:
        for man in range(n):
            if men_matchings[man] == 0:
                woman = mpref[man][last_rejected_by_rank_what[man]] - 1
                if woman + 1 == strategic_woman:
                    proposals_to_strategic_woman.add(man + 1)
                if women_matchings[woman] == 0:
                    men_matchings[man] = woman + 1
                    women_matchings[woman] = man + 1
                    curr_match_count += 1
                else:
                    curr_partner = women_matchings[woman]
                    curr_partner_rank = wpref[woman].index(curr_partner)
                    man_rank = wpref[woman].index(man + 1)
                    if curr_partner_rank < man_rank:
                        last_rejected_by_rank_what[man] = mpref[man].index(woman + 1) + 1
                    else:
                        men_matchings[curr_partner - 1] = 0
                        last_rejected_by_rank_what[curr_partner - 1] = mpref[curr_partner - 1].index(woman + 1) + 1
                        men_matchings[man] = woman + 1
                        women_matchings[woman] = man + 1
    if strategic_woman != 0:
        return men_matchings, proposals_to_strategic_woman
    else:
        return men_matchings

def generate_sw_manipulations(w_pref):
    manipulations = []
    for mi, mj in itertools.permutations(w_pref, 2):
        rest = [m for m in w_pref if m != mi and m != mj]
        new_pref = [mi, mj] + rest
        manipulations.append(new_pref)
    return manipulations

def generate_sm_manipulations(m_pref, mu_m):
    sm = []
    bm = [mu_m] + [w for w in m_pref if w != mu_m]
    sm.append(bm)
    for w0 in bm[1:]:
        manipulated = [w0] + [w for w in bm if w != w0]
        sm.append(manipulated)
    return sm

def two_for_one(mpref, wpref, accomplice, beneficiary, initial_matching):
    mpref_prime = copy.deepcopy(mpref)
    wpref_prime = copy.deepcopy(wpref)
    best_matching = initial_matching

    accomplice_pref = mpref[accomplice - 1]
    beneficiary_pref = wpref[beneficiary - 1]

    Sw = generate_sw_manipulations(beneficiary_pref)

    for w_misreport in Sw:
        Sm = generate_sm_manipulations(accomplice_pref, initial_matching[accomplice - 1])
        for m_misreport in Sm:
            mpref_prime[accomplice - 1] = m_misreport
            wpref_prime[beneficiary - 1] = w_misreport
            new_matching = gale_shapley(mpref_prime, wpref_prime)

            new_w_pairing = new_matching[beneficiary - 1]
            new_m_pairing = new_matching[accomplice - 1]

            if (beneficiary_pref.index(new_w_pairing) <
                    beneficiary_pref.index(best_matching[beneficiary - 1]) and
                    best_matching[accomplice - 1] == new_m_pairing):
                best_matching = new_matching
                accomplice_pref = m_misreport
                beneficiary_pref = w_misreport

    return best_matching, accomplice_pref, beneficiary_pref

# ----------------------------------------------------------------------
# 2.  Pool initializer + worker
# ----------------------------------------------------------------------
def _init_pool(shared_mpref, shared_wpref, shared_original):
    global MPREF, WPREF, ORIGINAL
    MPREF, WPREF, ORIGINAL = shared_mpref, shared_wpref, shared_original

def _pair_worker(pair):
    accomplice, beneficiary = pair
    best_matching, accomplice_pref_after, beneficiary_pref_after = two_for_one(
        MPREF, WPREF, accomplice, beneficiary, ORIGINAL
    )

    n = len(MPREF)
    rk1 = rk2 = 10**9
    partner1 = partner2 = None
    for i in range(n):
        if ORIGINAL[i] == beneficiary:
            partner1 = i + 1
        if best_matching[i] == beneficiary:
            partner2 = i + 1
    w_pref = WPREF[beneficiary - 1]
    for i in range(n):
        if w_pref[i] == partner1:
            rk1 = i
        if w_pref[i] == partner2:
            rk2 = i
    benefit = rk1 - rk2

    return (beneficiary, accomplice, benefit, best_matching,
            accomplice_pref_after, beneficiary_pref_after)

# ----------------------------------------------------------------------
# 3.  Utility: detect worker count from SLURM
# ----------------------------------------------------------------------
def detect_worker_count() -> int:
    for v in ("SLURM_CPUS_PER_TASK", "SLURM_NTASKS_PER_NODE", "SLURM_NTASKS"):
        if v in os.environ and os.environ[v].isdigit():
            return int(os.environ[v])
    return os.cpu_count() or 1

# ----------------------------------------------------------------------
# 4.  Main
# ----------------------------------------------------------------------
def run():
    csv_path = Path(CSV_NAME).resolve()
    if not csv_path.exists():
        sys.exit(f"[ERROR] CSV not found: {csv_path}")

    print(f"[INFO] Loading {csv_path.name} ...", flush=True)
    df = pd.read_csv(csv_path)

    if "preferences" not in df.columns or "original_matchings" not in df.columns:
        sys.exit("[ERROR] CSV must contain 'preferences' and 'original_matchings' columns.")
    df["preferences"] = df["preferences"].apply(ast.literal_eval)
    df["original_matchings"] = df["original_matchings"].apply(ast.literal_eval)

    df["two_for_one_results"] = None   # precreate the column
    out_path = Path(OUT_CSV_NAME).resolve()
    wall_start = time.time()

    workers = detect_worker_count()
    print(f"[INFO] Using {workers} worker processes\n", flush=True)

    for row_idx, row in df.iterrows():
        mpref, wpref = row["preferences"]
        original_matching = row["original_matchings"]

        n = len(mpref)
        pairs = list(itertools.product(range(1, n + 1), range(1, n + 1)))
        n_pairs = len(pairs)

        print(f"[row {row_idx + 1}] n={n} → {n_pairs:,} two_for_one calls", flush=True)

        processed = 0
        def progress():
            last = time.time()
            while processed < n_pairs:
                if time.time() - last >= 2:
                    print(f"[{time.strftime('%X')}] finished: {processed:,}/{n_pairs:,}", flush=True)
                    last = time.time()
                time.sleep(0.2)
            print(f"[{time.strftime('%X')}] finished: {processed:,}/{n_pairs:,}", flush=True)

        t_pr = threading.Thread(target=progress, daemon=True)
        t_pr.start()

        chunksz = max(1, n_pairs // (8 * workers))
        with mp.Pool(
            processes=workers,
            initializer=_init_pool,
            initargs=(mpref, wpref, original_matching),
        ) as pool:
            flat_results = []
            for tup in pool.imap_unordered(_pair_worker, pairs, chunksize=chunksz):
                flat_results.append(tup)
                processed += 1

        t_pr.join()

        instance_results = [[None for _ in range(n)] for _ in range(n)]
        for (beneficiary, accomplice, benefit, new_matching,
             accomplice_pref_after, beneficiary_pref_after) in flat_results:
            instance_results[beneficiary - 1][accomplice - 1] = [
                int(benefit),
                new_matching,
                accomplice_pref_after,
                beneficiary_pref_after
            ]

        # Save result for this row
        df.at[row_idx, "two_for_one_results"] = instance_results
        df.to_csv(out_path, index=False)   # write after every row
        print(f"[row {row_idx + 1}] results saved to {out_path}\n", flush=True)

    print(f"[INFO] All done. Final results in {out_path}")
    print(f"[INFO] Total wall-clock time: {time.time() - wall_start:.1f} s")

# ----------------------------------------------------------------------
# 5.  Entry-point
# ----------------------------------------------------------------------
if __name__ == "__main__":
    try:
        mp.set_start_method("fork", force=True)
    except RuntimeError:
        pass

    try:
        run()
    except KeyboardInterrupt:
        sys.stderr.write("\n[WARN] Interrupted by user — terminating.\n")
        sys.exit(1)


In [1]:
!jupyter nbconvert --to script two_for_one_gain_fast.ipynb


[NbConvertApp] Converting notebook two_for_one_gain_fast.ipynb to script
[NbConvertApp] Writing 9835 bytes to two_for_one_gain_fast.py
